# identity

> Deterministic node/edge identity: UUIDv5 over canonical identity tuples (stage-5 ratified rule: a node's id derives from what makes it THE same node across re-derivation, never from its correctable content).

In [ ]:
#| default_exp identity

In [ ]:
#| export
import uuid
from typing import Union

In [ ]:
#| export
# Fixed namespace for all context-graph layer ids. Derived once from the DNS
# namespace so the value is stable and reproducible; every deterministic id in
# the ecosystem hangs off this constant.
LAYER_ID_NAMESPACE = uuid.uuid5(uuid.NAMESPACE_DNS, "context-graph.cj-mills.com")

# Unit separator: cannot appear in paths/hashes/relation names, so joined
# identity parts can never collide across part boundaries.
IDENTITY_SEPARATOR = "\x1f"

In [ ]:
#| export
def canonical_part(
    value: Union[str, int, float],  # One identity-tuple part
) -> str:  # Canonical string form used inside the UUIDv5 name
    """Render one identity-tuple part canonically.

    Floats use `repr` (shortest round-trip — identical floats from the same
    deterministic computation render identically); ints use `str`; strings pass
    through. Anything else (including bool, whose int-ness is ambiguous) is
    rejected loudly: identity inputs must be deliberate.
    """
    if isinstance(value, bool):
        raise TypeError("bool is not a valid identity part (ambiguous int)")
    if isinstance(value, float):
        return repr(value)
    if isinstance(value, int):
        return str(value)
    if isinstance(value, str):
        return value
    raise TypeError(f"unsupported identity part type: {type(value).__name__}")

In [ ]:
#| export
def derive_node_id(
    kind: str,  # Node kind discriminator (e.g. "source", "audio-segment")
    *parts: Union[str, int, float],  # The identity tuple (positional, order-significant)
) -> str:  # Deterministic UUID string (UUIDv5)
    """Derive a deterministic node id from a kind + identity tuple.

    Same kind + same parts always yields the same id, across processes and
    re-derivations — re-derived graphs reproduce their node ids, so cross-graph
    references survive a rebuild (the G3a fix made structural). Content hashes
    belong in SourceRefs, NOT here: identity is position/provenance, never the
    correctable content.
    """
    name = IDENTITY_SEPARATOR.join([kind, *(canonical_part(p) for p in parts)])
    return str(uuid.uuid5(LAYER_ID_NAMESPACE, name))

In [ ]:
#| export
def derive_edge_id(
    source_id: str,      # Edge source node id
    target_id: str,      # Edge target node id
    relation_type: str,  # Relation type (e.g. "NEXT")
) -> str:  # Deterministic UUID string
    """Derive a deterministic edge id from (source, target, relation).

    Layer-0 structural edges are unique per (source, target, relation), so the
    triple IS the identity — re-derivation reproduces edge ids the same way it
    reproduces node ids.
    """
    return derive_node_id("edge", source_id, target_id, relation_type)

In [ ]:
# tests — determinism, distinctness, canonical parts
a = derive_node_id("source", "sha256:abc")
b = derive_node_id("source", "sha256:abc")
c = derive_node_id("source", "sha256:abd")
assert a == b, "same tuple -> same id"
assert a != c, "different tuple -> different id"
assert uuid.UUID(a).version == 5

# float canonicalization is repr-stable
assert canonical_part(300.2) == repr(300.2)
assert derive_node_id("audio-segment", a, 0.0, 300.2) == derive_node_id("audio-segment", a, 0.0, 300.2)

# part-boundary collisions impossible via the unit separator
assert derive_node_id("k", "ab", "c") != derive_node_id("k", "a", "bc")

# bools and odd types rejected
for bad in (True, None, [1]):
    try:
        canonical_part(bad)
        raise AssertionError("expected TypeError")
    except TypeError:
        pass

e1 = derive_edge_id("n1", "n2", "NEXT")
assert e1 == derive_edge_id("n1", "n2", "NEXT")
assert e1 != derive_edge_id("n2", "n1", "NEXT")
print("identity tests OK")